In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Clear filesystem cache (pagecache, dentries, inodes)
!sudo sync
!sudo sh -c 'echo 3 > /proc/sys/vm/drop_caches'

sh: 1: cannot create /proc/sys/vm/drop_caches: Read-only file system


In [3]:
# Check GPU status
!nvidia-smi

Mon Mar 30 17:43:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.18             Driver Version: 580.126.18     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:03:00.0 Off |                  N/A |
|  0%   48C    P8             27W /  350W |       0MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# For Qwen2.5-VL
# # %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     %pip install unsloth  # Do this in local & cloud setups
# else:
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
# %pip install transformers==4.56.2
# %pip install --no-deps trl==0.22.2

# For Qwen3-VL
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.57.1
%pip install --no-deps trl==0.22.2

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 79.0 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 44.1 MB/s  0:00:17m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 47.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 47.8 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 38.6 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 41.7 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 42.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 24.4 MB/s  0:00:22m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 22.7 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22

# Configuration

In [5]:
# Fix TorchDynamo bug
import os
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

In [20]:
SEED = 42

# Model configuration
# MODEL_ID = 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'
MODEL_ID = 'unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit'

# Data configuration
SFT_DATA_ID = 'jxie/coco_captions'
DATA_SPLIT = 'train'
DATA_SIZE = 1250
TEST_RATIO = 0.1

BVIR_DATA_ID = 'alxxtexxr/BIVR-Data'
# CODEBOOK_PATH = 'model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy'
CODEBOOK_PATH = 'model_unsloth_Qwen3-VL-8B-Instruct-unsloth-bnb-4bit.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy'

# Data

In [7]:
from datasets import load_dataset, Dataset

def load_hf_dataset(data_id, split, total_size):
    dataset_stream = load_dataset(data_id, split=split, streaming=True)
    
    unique_cocoids = set()
    sliced_data = []
    for example in dataset_stream:
        if len(sliced_data) >= total_size:
            break

        cocoid = example.get('cocoid', None)
        if cocoid is not None:
            if cocoid in unique_cocoids:
                continue
            unique_cocoids.add(cocoid)

        sliced_data.append(example)

    dataset = Dataset.from_list(sliced_data)
    return dataset

# Load all split sets
train_set = load_hf_dataset(SFT_DATA_ID, split='train', total_size=int(DATA_SIZE*(1-TEST_RATIO*2)))
val_set = load_hf_dataset(SFT_DATA_ID, split='validation', total_size=int(DATA_SIZE*TEST_RATIO))
test_set = load_hf_dataset(SFT_DATA_ID, split='test', total_size=int(DATA_SIZE*TEST_RATIO))

print("Train dataset:")
print(train_set)
print("\nValidation dataset:")
print(val_set)
print("\nTest dataset:")
print(test_set)

README.md:   0%|          | 0.00/626 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

Train dataset:
Dataset({
    features: ['image', 'filename', 'cocoid', 'caption'],
    num_rows: 1000
})

Validation dataset:
Dataset({
    features: ['image', 'filename', 'cocoid', 'caption'],
    num_rows: 125
})

Test dataset:
Dataset({
    features: ['image', 'filename', 'cocoid', 'caption'],
    num_rows: 125
})


In [8]:
from datasets import concatenate_datasets

# Add a 'source' column to each split set
train_set = train_set.add_column('source', ['train'] * len(train_set))
val_set   = val_set.add_column('source', ['val'] * len(val_set))
test_set  = test_set.add_column('source', ['test'] * len(test_set))

# Combine all split sets into one dataset
dataset = concatenate_datasets([train_set, val_set, test_set])
print(dataset)

Dataset({
    features: ['image', 'filename', 'cocoid', 'caption', 'source'],
    num_rows: 1250
})


In [9]:
from PIL import Image

def process_image(example):
    image_col = 'decoded_image' if 'decoded_image' in example else 'image'
    image = example[image_col]
    image = image.resize((512, 512), Image.LANCZOS)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    example['decoded_image'] = image
    return example

# # Resize to (512, 512) and convert to RGB
# train_set = train_set.map(process_image)
# val_set = val_set.map(process_image)
# test_set = test_set.map(process_image)

# # Remove the original 'image' column
# train_set = train_set.remove_columns('image')
# val_set = val_set.remove_columns('image')
# test_set = test_set.remove_columns('image')

# # Rename the 'decoded_image' column to 'image'
# train_set = train_set.rename_column('decoded_image', 'image')
# val_set = val_set.rename_column('decoded_image', 'image')
# test_set = test_set.rename_column('decoded_image', 'image')

# print("Train dataset:")
# print(train_set)
# print("\nValidation dataset:")
# print(val_set)
# print("\nTest dataset:")
# print(test_set)

dataset = dataset.map(process_image) # Resize to (512, 512) and convert to RGB
dataset = dataset.remove_columns('image') # Remove the original 'image' column
dataset = dataset.rename_column('decoded_image', 'image') # Rename the 'decoded_image' column to 'image'
print(dataset)

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

Dataset({
    features: ['filename', 'cocoid', 'caption', 'source', 'image'],
    num_rows: 1250
})


# Model

In [10]:
from unsloth import FastVisionModel
import torch
from transformers import AutoProcessor

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16-bit LoRA.
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

# Visual Codebook

In [11]:
from huggingface_hub import hf_hub_download

codebook_path = hf_hub_download(
    repo_id=BVIR_DATA_ID,
    filename=CODEBOOK_PATH,
    repo_type='dataset',
)
print("Visual codebook downloaded to:", codebook_path)

model_unsloth_Qwen3-VL-8B-Instruct-unslo(…):   0%|          | 0.00/134M [00:00<?, ?B/s]

Visual codebook downloaded to: /root/.cache/huggingface/hub/datasets--alxxtexxr--BIVR-Data/snapshots/ce125fc8cf6c125a44aad9685a92ac59448f4fcf/model_unsloth_Qwen3-VL-8B-Instruct-unsloth-bnb-4bit.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy


In [13]:
# %%capture
%pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 78.5 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [14]:
import numpy as np
import faiss

codebook = np.load(codebook_path).astype(np.float32) # (K, 3584) -> Current: (8192, 3584)
faiss.normalize_L2(codebook)

print("Visual codebook shape:", codebook.shape)
print("Mean L2 norm (should be ~1):", np.linalg.norm(codebook, axis=1).mean())

Visual codebook shape: (8192, 4096)
Mean L2 norm (should be ~1): 1.0


In [15]:
d = codebook.shape[1] # Current: 3584

# FAISS search (L2 is equivalent to cosine for unit vectors)
index = faiss.IndexFlatL2(d)
index.add(codebook)

# Image Serialization

In [16]:
distances_all = []

def serialize_img(example, idx):
    print(f"Processing example #{idx}")
    
    image = example['image']
    inputs = processor.image_processor(images=[image], return_tensors='pt')
    pixel_values = inputs['pixel_values'].to(model.device)     # Current: (1296, 1176)
    image_grid_thw = inputs['image_grid_thw'].to(model.device) # Current: (1, 3)
    
    # Get visual embeddings of the image from the model
    with torch.no_grad():
        visual_embs = model.visual(pixel_values, image_grid_thw) # Current: (324, 3584)
    
    # Take the first element as visual embeddings if it's a tuple (for compatibility with different model versions)
    if isinstance(visual_embs, tuple):
        visual_embs = visual_embs[0]
        
    visual_embs = visual_embs.cpu().float().numpy() # Convert to Float32 NumPy array
    faiss.normalize_L2(visual_embs)                 # Normalize to unit length
    
    # Perform FAISS search to find nearest codebook vector for each visual embedding
    distances, codebook_idxs = index.search(visual_embs, 1)
    distances, codebook_idxs = distances.flatten(), codebook_idxs.flatten()
    # distances     -> Current: (324,)
    # codebook_idxs -> Current: (324,)
    distances_all.extend(distances)
    
    # Serialize image into a string format using the codebook indices (e.g., "<|vtok_123|><|vtok_456|>...")
    image_serialized = "".join([f'<|vtok_{i}|>' for i in codebook_idxs])
    example['image_serialized'] = image_serialized
    del visual_embs
    return example

dataset = dataset.map(serialize_img, with_indices=True)

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

Processing example #0
Processing example #1
Processing example #2
Processing example #3
Processing example #4
Processing example #5
Processing example #6
Processing example #7
Processing example #8
Processing example #9
Processing example #10
Processing example #11
Processing example #12
Processing example #13
Processing example #14
Processing example #15
Processing example #16
Processing example #17
Processing example #18
Processing example #19
Processing example #20
Processing example #21
Processing example #22
Processing example #23
Processing example #24
Processing example #25
Processing example #26
Processing example #27
Processing example #28
Processing example #29
Processing example #30
Processing example #31
Processing example #32
Processing example #33
Processing example #34
Processing example #35
Processing example #36
Processing example #37
Processing example #38
Processing example #39
Processing example #40
Processing example #41
Processing example #42
Processing example #4

In [17]:
from datasets import DatasetDict

# Split the dataset back into train, val, test sets based on the 'source' column
train_set = dataset.filter(lambda x: [s == 'train' for s in x['source']], batched=True)
val_set   = dataset.filter(lambda x: [s == 'val' for s in x['source']], batched=True)
test_set  = dataset.filter(lambda x: [s == 'test' for s in x['source']], batched=True)

# Create a DatasetDict for the split sets
dataset_split = DatasetDict({
    'train': train_set,
    'val': val_set,
    'test': test_set,
})
print(dataset_split)

Filter:   0%|          | 0/1250 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1250 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1250 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['filename', 'cocoid', 'caption', 'source', 'image', 'image_serialized'],
        num_rows: 1000
    })
    val: Dataset({
        features: ['filename', 'cocoid', 'caption', 'source', 'image', 'image_serialized'],
        num_rows: 125
    })
    test: Dataset({
        features: ['filename', 'cocoid', 'caption', 'source', 'image', 'image_serialized'],
        num_rows: 125
    })
})


In [21]:
# Remove 'source' column and shuffle for each split set
dataset_split = DatasetDict({
    split: ds.remove_columns('source').shuffle(seed=SEED)
    for split, ds in dataset_split.items()
})
print(dataset_split)

DatasetDict({
    train: Dataset({
        features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
        num_rows: 1000
    })
    val: Dataset({
        features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
        num_rows: 125
    })
    test: Dataset({
        features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
        num_rows: 125
    })
})


In [22]:
# Compute RMSE
rmse = np.sqrt(np.vstack(distances_all).mean())
print("RMSE:", rmse)

RMSE: 0.65477586


In [26]:
# Push the final dataset to Hugging Face Hub
dataset_split.push_to_hub(f"alxxtexxr/{SFT_DATA_ID.split('/')[-1]}_{DATA_SIZE/1000}k_serialized_v260331")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/coco_captions_1.25k_serialized_v260331/commit/85d62b64e64cb55c1c7b295539b76ef3fb24cc60', commit_message='Upload dataset', commit_description='', oid='85d62b64e64cb55c1c7b295539b76ef3fb24cc60', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/coco_captions_1.25k_serialized_v260331', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/coco_captions_1.25k_serialized_v260331'), pr_revision=None, pr_num=None)